> 在线讲义：[第 60 章 · 强化学习训练系统：生成、验证、学习组成的反馈循环](https://wikieden.github.io/ai-infra-from-constraints/lessons/60-rl-system.html) · 本书《从约束推导 AI Infra》

# 第 60 章 · 强化学习训练系统：生成、验证、学习组成的反馈循环

配套讲义：仓库里 `ai-infra/notes/lessons/60-rl-system.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



组内优势、三阶段有效吞吐、共享加速器切换次序、按分片同步权重、概率比、异步周期与路由重放记录，取自 AII 第 10 章 §10.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `advantages`**：小工具。

In [ ]:
def advantages(rewards: list[float]) -> list[float]:
    m = sum(rewards) / len(rewards)
    return [r - m for r in rewards]

**函数 `effective_rate`**：r_eff = min(r_l, q · min(r_g, r_v))。

In [ ]:
def effective_rate(rg: float, rv: float, rl: float, q: float) -> float:
    """r_eff = min(r_l, q · min(r_g, r_v))。"""
    return min(rl, q * min(rg, rv))

**函数 `switch_peak`**：共享加速器从训练切到生成的显存峰值（GiB）。

In [ ]:
def switch_peak(order: str, train: float = 40, weights: float = 15.3, kv: float = 24, common: float = 4) -> float:
    """共享加速器从训练切到生成的显存峰值（GiB）。"""
    if order == "加载权重和 KV 再释放训练":
        return train + weights + kv + common
    return train + weights + common          # 先加载权重，释放训练后再分配 KV

**函数 `ratio`**：小工具。

In [ ]:
def ratio(logp_theta: float, logp_mu: float) -> float:
    return math.exp(logp_theta - logp_mu)

**函数 `async_gain`**：异步有效吞吐 / 同步有效吞吐 = (q / (max + sync)) / (1 / (gen + learn + sync))。

In [ ]:
def async_gain(q: float, gen: float = 40, learn: float = 16, sync: float = 4) -> float:
    """异步有效吞吐 / 同步有效吞吐 = (q / (max + sync)) / (1 / (gen + learn + sync))。"""
    return q / (max(gen, learn) + sync) * (gen + learn + sync)

**函数 `survive`**：小工具。

In [ ]:
def survive(lam: float, t: float) -> float:
    return math.exp(-lam * t)

**函数 `replay_bytes`**：小工具。

In [ ]:
def replay_bytes(tokens: int = 8192, layers: int = 48, k: int = 8, width: int = 2) -> int:
    return tokens * layers * k * width

**函数 `top2`**：小工具。

In [ ]:
def top2(scores: list[float]) -> list[int]:
    return sorted(range(len(scores)), key=lambda i: -scores[i])[:2]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：组内优势")
print(f"  奖励 [1, 1] → {advantages([1, 1])}；[1, 0] → {advantages([1, 0])}")

In [ ]:
print("\n第二步：三阶段有效吞吐（条/s）")
print(f"  生成 12、验证 6、学习 8、保留 75% → {effective_rate(12, 6, 8, 0.75)}")
print(f"  生成翻倍 → {effective_rate(24, 6, 8, 0.75)}；验证翻倍 → {effective_rate(12, 12, 8, 0.75)}（学习成瓶颈）")

In [ ]:
print("\n第三步：切换次序与权重同步")
for o in ("加载权重和 KV 再释放训练", "先加载权重，释放后再分配 KV"):
    print(f"  {o}：峰值 {switch_peak(o):.1f} GiB（H100 74.5 GiB）")
print(f"  Qwen3-235B 专家 423 GiB、EP16：整份发给每个接收方 {16 * 423} GiB，按分片 423 GiB（每方 {423 / 16:.1f} GiB）")

In [ ]:
print("\n第四步：策略版本与异步")
print(f"  logprob −3 vs 误记 −3.25：比值 {ratio(-3, -3.25):.2f}")
print(f"  同步周期 60 s、异步 44 s：q 门槛 {44 / 60 * 100:.1f}%；q = 0.8 → {async_gain(0.8):.2f} 倍，q = 0.7 → {async_gain(0.7):.2f} 倍")
print(f"  中断率每小时 0.1 次：10 分钟轨迹完整概率 {survive(0.1 / 3600, 600):.3f}，2 小时 {survive(0.1 / 3600, 7200):.3f}")

In [ ]:
print("\n第五步：路由重放")
print(f"  top-2：[0.500, 0.301, 0.300] → {top2([0.5, 0.301, 0.3])}；第三个变 0.302 → {top2([0.5, 0.301, 0.302])}")
print(f"  Qwen3-30B-A3B 8192 token × 48 层 × top-8：uint16 {replay_bytes() / 2**20:.0f} MiB，int32 {replay_bytes(width=4) / 2**20:.0f} MiB")

## 3. 代码里才看得到的

1. `async_gain` 等于 $q \cdot 60/44$，所以门槛就是 $44/60$，与 batch 大小无关。
2. `top2` 用的是 0 起编号，[0, 2] 对应原文的"专家 1、3"。
3. `survive` 里中断率按每小时 0.1 次是演示值，原文没给具体数。

## 4. 自己改一改

- 给 `effective_rate` 加第四道工序（教师前向），看 OPD 的瓶颈。
- 写一个函数：给定 KV 池大小，求切换峰值不超过容量的最大训练状态。
- 用 `survive` 算：要让 90% 的轨迹不被中断，轨迹最长能多久。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 学生自己做题老师批，做完一遍不等于学会；
> 三道工序看最慢的，加人先加检查员；
> 小房间换布置，先搬大件、再清旧的、最后摆小件；
> 旧说明书的零件扔太多，提前开工就亏；
> 岔路口记下当时走哪条，训练照着走。